# 01 Data Exploration

This notebook explores the processed Lending Club dataset and covers:

- Basic dataset statistics
- Sample distribution of the 5-class risk label
- Distribution analysis for key continuous variables
- Figure export to `results/figures/`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['axes.unicode_minus'] = False

# Automatically locate the project root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / 'train.csv'
VAL_PATH = PROCESSED_DIR / 'val.csv'
TEST_PATH = PROCESSED_DIR / 'test.csv'

for required_path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing processed file: {required_path}')

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
val_df = pd.read_csv(VAL_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

risk_name_map = {
    0: 'Normal',
    1: 'Watchlist',
    2: 'Substandard',
    3: 'Doubtful',
    4: 'Loss',
}

label_df = pd.concat([
    train_df[['preloan_risk_label']],
    val_df[['preloan_risk_label']],
    test_df[['preloan_risk_label']],
], ignore_index=True)
label_df['risk_display_name'] = label_df['preloan_risk_label'].map(risk_name_map)

display(train_df.head())


In [ ]:
# Basic dataset statistics
print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)

print('\nTrain info:')
train_df.info()

summary_df = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'missing_rate': train_df.isna().mean().round(4),
    'nunique': train_df.nunique(dropna=True),
}).sort_values(by=['missing_rate', 'nunique'], ascending=[False, False])

summary_df.head(20)


In [ ]:
# Risk label distribution
risk_dist = (
    label_df.groupby(['preloan_risk_label', 'risk_display_name'])
    .size()
    .reset_index(name='count')
    .sort_values('preloan_risk_label')
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=risk_dist,
    x='risk_display_name',
    y='count',
    hue='risk_display_name',
    palette='Blues_d',
    legend=False,
)
ax.set_title('Risk Label Distribution')
ax.set_xlabel('Risk Label')
ax.set_ylabel('Sample Count')
for patch, count in zip(ax.patches, risk_dist['count']):
    ax.annotate(
        f'{int(count):,}',
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha='center',
        va='bottom',
        fontsize=10,
        xytext=(0, 5),
        textcoords='offset points',
    )
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'risk_label_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

risk_dist


In [ ]:
# Continuous variable distributions
continuous_columns = ['loan_amnt', 'annual_inc', 'dti', 'calibrated_credit_limit']
available_columns = [col for col in continuous_columns if col in train_df.columns]

# Use stable sampling to keep plotting responsive.
plot_df = train_df[available_columns].dropna().copy()
if len(plot_df) > 50000:
    plot_df = plot_df.sample(50000, random_state=42)

fig, axes = plt.subplots(len(available_columns), 2, figsize=(14, 4 * len(available_columns)))
if len(available_columns) == 1:
    axes = [axes]

for idx, column in enumerate(available_columns):
    hist_ax, box_ax = axes[idx]
    sns.histplot(plot_df[column], kde=True, ax=hist_ax, color='#4C72B0')
    hist_ax.set_title(f'{column} Histogram')

    sns.boxplot(x=plot_df[column], ax=box_ax, color='#55A868')
    box_ax.set_title(f'{column} Boxplot')

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'continuous_variable_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved figures to:', FIGURE_DIR)
